# 🎬 KINO Kaggle-motor (egen GPU) – reserv när Colab inte ansluter

Samma motor som Colab-versionen (samma server, samma endpoints och samma säkerhetsregler: minderårsspärr, åldersspärr 21 på ansikten, ansiktsbyte bara mellan era egna artister med samtycke, NSFW bara där KINO tillåter det). Servern hämtas från GitHub (`colab/kino_colab_server.py`), så rättningar gäller båda.

**Så gör du:**
1. kaggle.com → **Create → New Notebook** → **File → Import Notebook** → klistra in `https://github.com/pistpejl/kino/blob/main/kaggle/kino_kaggle_engine.ipynb`.
2. Högerpanelen **Settings**: **Accelerator → GPU T4 x2**, **Internet → On** (kräver telefonverifierat Kaggle-konto).
3. *Valfritt:* **Add-ons → Secrets** → lägg till `HF_TOKEN` (bara för FLUX 3 Image Klein, som är gated).
4. **Run All**. Sista cellerna visar **adress** och **nyckel** i en grön ruta.
5. I KINO: **Colab / Kaggle (egen GPU)** → klistra in adress + nyckel → **Spara & testa**.

Kaggle ger ca 30 GPU-timmar/vecka och max 12 h per session. Stäng sessionen (Stop) när du är klar.

## 1. Kolla GPU

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # T4 x2: motorn använder bara den första (cuda:0)
os.environ['KINO_PLATFORM'] = 'kaggle'
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'INGEN GPU: Settings → Accelerator → GPU T4 x2'
import torch
assert torch.cuda.is_available(), 'Ingen GPU. Välj Settings → Accelerator → GPU T4 x2 och kör igen.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)
import urllib.request
try: urllib.request.urlopen('https://huggingface.co', timeout=10); print('Internet: på')
except Exception: raise SystemExit('Internet är av: Settings → Internet → On (kräver telefonverifiering).')

## 2. Installera (låsta versioner; Kaggles egen torch/CUDA används och installeras inte om)

In [ ]:
!pip -q install diffusers==0.42.0 transformers==5.20.2 accelerate==1.15.0 bitsandbytes==0.50.2 spandrel==0.4.2 sentencepiece==0.2.1 ftfy==6.3.1 imageio==2.37.0 imageio-ffmpeg==0.6.0 \
    fastapi==0.115.12 uvicorn==0.34.2 python-multipart==0.0.20 librosa==0.11.0 soundfile==0.13.1 openai-whisper==20250625 opencv-python-headless==4.11.0.86 insightface==0.7.3 onnxruntime-gpu==1.20.2
!pip -q install --no-deps demucs==4.0.1 dora-search==0.1.12 julius==0.2.7 lameenc==1.8.1 openunmix==1.3.0 && pip -q install einops==0.8.1 treetable==0.2.5 submitit==1.5.2 omegaconf==2.3.0 retrying==1.3.4
!apt-get -qq install -y ffmpeg > /dev/null
# Cloudflare quick tunnel (gratis, inget konto), låst version från Cloudflares officiella GitHub
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/download/2025.4.0/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print('Klart')

## 3. Hämta servern från GitHub (en källa för Colab och Kaggle)

In [ ]:
import urllib.request, os
os.makedirs('/kaggle/working', exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/pistpejl/kino/main/colab/kino_colab_server.py', '/kaggle/working/kino_colab_server.py')
print('Server hämtad:', os.path.getsize('/kaggle/working/kino_colab_server.py'), 'byte')

## 4. (Valfritt) Ansiktsbyte
Hämtar inswapper_128.onnx och kontrollerar SHA-256. Misslyckas det: lägg filen som Kaggle-dataset (Add Data) och kopiera den till `/kaggle/working/models/`. Endast icke-kommersiellt bruk.

In [ ]:
import hashlib, os, urllib.request
os.makedirs('/kaggle/working/models', exist_ok=True)
P = '/kaggle/working/models/inswapper_128.onnx'
SHA = 'e4a3f08c753cb72d04e10aa0f7dbe3deebbf39567d4ead6dce08e98aa49e16af'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
if not os.path.exists(P):
    try: urllib.request.urlretrieve('https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx', P)
    except Exception as e: print('Nedladdning misslyckades:', e)
if os.path.exists(P) and sha(P) != SHA:
    os.remove(P); print('Fel kontrollsumma: filen raderades.')
if not os.path.exists(P):
    import glob  # Kaggle har ingen uppladdningsruta: leta i tillagda dataset (Add Data)
    for c in glob.glob('/kaggle/input/**/inswapper_128.onnx', recursive=True):
        if sha(c) == SHA: import shutil; shutil.copy(c, P); break
print('Ansiktsbyte redo:', os.path.exists(P))

## 5. (Valfritt) Lipsync med LatentSync 1.5 (ca 10 min)

In [ ]:
import os, subprocess
def sh(c): print('$', c[:100]); subprocess.run(c, shell=True, check=True)
if not os.path.isdir('/kaggle/working/LatentSync'):
    sh('git clone -q https://github.com/bytedance/LatentSync /kaggle/working/LatentSync')
sh('cd /kaggle/working/LatentSync && git checkout -q a229c3948406bc2cf6eaf4873e662e70c6a04746')
sh('apt-get -qq install -y libgl1 > /dev/null; pip -q install uv==0.7.3')
if not os.path.exists('/kaggle/working/ls_env/bin/python'):
    sh('uv venv /kaggle/working/ls_env --python 3.10 --seed')
    sh('uv pip install --python /kaggle/working/ls_env/bin/python -r /kaggle/working/LatentSync/requirements.txt')
from huggingface_hub import hf_hub_download
for f in ['latentsync_unet.pt', 'whisper/tiny.pt']:
    hf_hub_download('ByteDance/LatentSync-1.5', f, local_dir='/kaggle/working/LatentSync/checkpoints')
print('LatentSync 1.5 klar.')

## (Valfritt) Röstbyte med Seed-VC (ca 5 min)
Officiella repot `Plachtaa/seed-vc`, låst till commit `51383efd9210`, med sångmodellen (whisper-base, f0, 44,1 kHz) från Hugging Face `Plachta/Seed-VC`. Körs i en egen Python 3.10-miljö med egna låsta versioner (Seed-VC kräver transformers 4.46.3 och påverkar därför inte motorns diffusers 0.42.0 / transformers 5.20.2 / accelerate 1.15.0). **Bara röster ur din samtyckta röstlista** kan användas. Seed-VC är GPL-3.0.

In [ ]:
import os, subprocess
def sh(c): print('$', c[:100]); subprocess.run(c, shell=True, check=True)
if not os.path.isdir('/kaggle/working/seed-vc'):
    sh('git clone -q https://github.com/Plachtaa/seed-vc /kaggle/working/seed-vc')
sh('cd /kaggle/working/seed-vc && git checkout -q 51383efd921027683c89e5348211d93ff12ac2a8')
sh('pip -q install uv==0.7.3')
if not os.path.exists('/kaggle/working/svc_env/bin/python'):
    sh('uv venv /kaggle/working/svc_env --python 3.10 --seed')
    sh('uv pip install --python /kaggle/working/svc_env/bin/python torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 numpy==1.26.4 scipy==1.13.1 librosa==0.10.2 soundfile==0.12.1 huggingface-hub==0.28.1 munch==4.0.0 einops==0.8.0 descript-audio-codec==1.0.0 pydub==0.25.1 transformers==4.46.3 accelerate==1.0.1 hydra-core==1.3.2 pyyaml==6.0.2 tqdm==4.67.1')
# Hamtar vikterna en gang (kort testkonvertering med repots exempelfiler)
sh('cd /kaggle/working/seed-vc && /kaggle/working/svc_env/bin/python inference.py --source examples/source/source_s1.wav --target examples/reference/s1p1.wav '
   '--output /tmp/svc_test --diffusion-steps 10 --f0-condition True --fp16 True > /tmp/svc_test.log 2>&1 || tail -20 /tmp/svc_test.log')
print('Seed-VC klar.' if os.path.isdir('/tmp/svc_test') else 'Seed-VC: testet misslyckades, se loggen ovan.')

## 6. Starta server + publik tunnel

In [ ]:
import os, secrets, sys, threading, subprocess, re, time, uvicorn
from IPython.display import display, HTML
os.environ['KINO_TOKEN'] = secrets.token_urlsafe(24)   # ny slumpad token varje körning
os.environ['KINO_PLATFORM'] = 'kaggle'; os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
try:  # valfritt: HF_TOKEN i Add-ons → Secrets, bara för FLUX 3 Image Klein (gated). Skrivs aldrig ut.
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception: pass
sys.path.insert(0, '/kaggle/working')
import kino_colab_server as kino
threading.Thread(target=lambda: uvicorn.run(kino.app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
time.sleep(3)
cf = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in cf.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()
display(HTML(f'''<div style="font-family:sans-serif;padding:16px;border:3px solid #2a2;border-radius:12px">
<div style="font-size:16px">URL (klistra in i KINO):</div><div style="font-size:26px;font-weight:bold;user-select:all">{url}</div>
<div style="font-size:16px;margin-top:12px">Token:</div><div style="font-size:22px;font-weight:bold;user-select:all">{kino.TOKEN}</div></div>'''))
print('URL:  ', url); print('TOKEN:', kino.TOKEN)

## 7. Håll igång
Kör kvar så länge du använder motorn. Stop/stäng sessionen = motorn stängs.

In [ ]:
while True:
    print(time.strftime('%H:%M'), '| kö:', kino.q.qsize(), '| laddad:', kino._cur['name'], '| artister:', len(kino.roster),
          '| senaste jobb:', {k: v['kind'] + ':' + v['status'] for k, v in list(kino.jobs.items())[-4:]})
    time.sleep(60)